In [1]:
import pandas as pd

In [2]:
# 1. Chargement des deux fichiers
df_campaign = pd.read_csv("../data/outputs/final_campaign_offers.csv")
df_enriched = pd.read_csv("../data/processed/df_enriched_final.csv")

# 2. Création du référentiel de prix (on prend le prix unique par nom de produit)
# On garde product_name et unit_price
prices_ref = df_enriched[['product_name', 'unit_price']].drop_duplicates(subset='product_name')

# 3. Fusion pour obtenir le prix du produit RECOMMANDÉ (product_name_right)
df_impact = df_campaign.merge(
    prices_ref,
    left_on='product_name_right',
    right_on='product_name',
    how='left'
)

# Nettoyage : on supprime la colonne 'product_name' redondante du merge
df_impact = df_impact.drop(columns=['product_name_y']).rename(columns={'product_name_x': 'product_name'})

# Vérification des prix manquants
missing_prices = df_impact['unit_price'].isnull().sum()
if missing_prices > 0:
    print(f"⚠️ {missing_prices} produits n'ont pas de prix. Remplissage par la moyenne.")
    df_impact['unit_price'] = df_impact['unit_price'].fillna(df_impact['unit_price'].mean())

MemoryError: Unable to allocate 2.52 GiB for an array with shape (10, 33819106) and data type int64

In [ ]:
def calculate_financial_impact(df):
    # --- SECTION 7A : Uplift ---
    uplift_logic = {"10%": 1.15, "20%": 1.35}
    df['uplift_factor'] = df['discount_value'].map(uplift_logic)
    df['prob_with_promo'] = (df['confidence'] * df['uplift_factor']).clip(upper=1.0)
    df['incremental_sales'] = df['prob_with_promo'] - df['confidence']

    # --- SECTION 7B : Incremental Revenue ---
    df['incremental_revenue'] = df['incremental_sales'] * df['unit_price']

    # --- SECTION 7C : Discount Cost ---
    # Conversion du discount string ("20%") en float (0.2)
    df['discount_rate'] = df['discount_value'].str.replace('%', '').astype(float) / 100
    # Coût appliqué à tous ceux qui achètent avec la promo
    df['discount_cost'] = df['prob_with_promo'] * df['unit_price'] * df['discount_rate']

    # --- SECTION 7D : Net Impact ---
    df['net_profit_impact'] = df['incremental_revenue'] - df['discount_cost']

    return df

# Exécution
df_final = calculate_financial_impact(df_impact)

# Affichage pour contrôle
print(df_final[['product_name_right', 'unit_price', 'incremental_revenue', 'net_profit_impact']].head())

In [ ]:
# Conversion du discount en valeur numérique (si pas déjà fait)
df_final['discount_rate'] = df_final['discount_value'].str.replace('%', '').astype(float) / 100

# 7C — Discount Cost
# On paie le discount sur tous les acheteurs prévus (Base + Uplift)
df_final['discount_cost'] = df_final['prob_with_promo'] * df_final['unit_price'] * df_final['discount_rate']

# 7D — Net Impact (Le ROI)
df_final['net_profit'] = df_final['incremental_revenue'] - df_final['discount_cost']

In [ ]:
# Correction : on ne garde que 70% de l'incrémental
df_final['true_incremental_sales'] = df_final['incremental_sales'] * 0.7
df_final['true_incremental_revenue'] = df_final['true_incremental_sales'] * df_final['unit_price']

# Le profit corrigé (Le coût du discount reste le même !)
df_final['true_net_profit'] = df_final['true_incremental_revenue'] - df_final['discount_cost']

In [ ]:
# Estimation simple de la réduction du churn (Valeur de rétention)
df_final['churn_savings'] = df_final['incremental_sales'] * (df_final['unit_price'] * 0.5) # On estime 50% du prix comme valeur de rétention

In [ ]:
# Agrégation finale
final_dashboard = df_final.groupby('cluster').agg({
    'user_id': 'count',
    'incremental_sales': 'sum',
    'incremental_revenue': 'sum',
    'discount_cost': 'sum',
    'net_profit': 'sum',
    'true_net_profit': 'sum'
}).rename(columns={'user_id': 'clients_ciblés'})

# Calcul de la rentabilité par client
final_dashboard['ROI_per_client'] = final_dashboard['true_net_profit'] / final_dashboard['clients_ciblés']

# Affichage propre
print("✅ RÉPONSE BUSINESS DIRECTE :")
display(final_dashboard.round(2).sort_values(by='true_net_profit', ascending=False))

In [ ]:
# --- ÉTAPE FINALE : COMPILATION ET EXPORT ---

def finalize_business_report(df):
    report = df.copy()

    # 1. Rappel des calculs financiers (Sections 7A -> 7E)
    report['incremental_sales'] = report['prob_with_promo'] - report['confidence']
    report['incremental_revenue'] = report['incremental_sales'] * report['unit_price']

    report['discount_rate'] = report['discount_value'].str.replace('%', '').astype(float) / 100
    report['discount_cost'] = report['prob_with_promo'] * report['unit_price'] * report['discount_rate']

    report['net_profit'] = report['incremental_revenue'] - report['discount_cost']

    # Correction de cannibalisation (Senior Insight 7E)
    report['true_net_profit'] = (report['incremental_sales'] * 0.7 * report['unit_price']) - report['discount_cost']

    # 2. Application des décisions (Le "Cerveau" du Directeur)
    def get_decision(row):
        if row['true_net_profit'] > 0:
            return "✅ GO"
        elif row['true_net_profit'] > -0.10:
            return "⚠️ TEST"
        else:
            return "❌ NO GO"

    report['decision'] = report.apply(get_decision, axis=1)

    # 3. Sélection et réorganisation des colonnes pour le CSV
    final_cols = [
        'user_id', 'cluster', 'product_name', 'product_name_right',
        'unit_price', 'discount_value', 'confidence', 'prob_with_promo',
        'incremental_revenue', 'discount_cost', 'true_net_profit', 'decision'
    ]

    return report[final_cols]

# Génération du rapport
df_final_report = finalize_business_report(df_final)

# Export en CSV
output_path = "../data/outputs/business_roi_report_final.csv"
df_final_report.to_csv(output_path, index=False, sep=',', encoding='utf-8')

print(f"✅ Rapport généré avec succès ici : {output_path}")
print(f"📊 Nombre d'offres analysées : {len(df_final_report)}")
print(f"💰 Profit net total estimé : {df_final_report['true_net_profit'].sum():.2f}€")

In [ ]:
# 1. Créer un référentiel des départements par produit
# On utilise df_enriched que vous avez chargé précédemment
dept_ref = df_enriched[['product_name', 'department']].drop_duplicates(subset='product_name')

# 2. Fusionner avec votre table de campagne
# On veut savoir de quel département vient le produit recommandé (B)
df_final = df_final.merge(
    dept_ref,
    left_on='product_name_right',
    right_on='product_name',
    how='left'
)

# 3. Nettoyage (on enlève la colonne product_name en double créée par le merge)
if 'product_name_y' in df_final.columns:
    df_final = df_final.drop(columns=['product_name_y']).rename(columns={'product_name_x': 'product_name'})
elif 'product_name' in df_final.columns and 'product_name_right' in df_final.columns:
    # Si le merge a créé un doublon product_name, on s'assure de garder la structure propre
    pass

# Vérification :
print(f"Colonnes disponibles : {df_final.columns.tolist()}")

#1. Configuration des Marges par Département (Benchmarks Retail)
margins_config = {
    'alcohol': 0.40, 'personal care': 0.35, 'household': 0.30,
    'produce': 0.35, 'meat seafood': 0.25, 'pantry': 0.25,
    'snacks': 0.25, 'beverages': 0.25, 'frozen': 0.25,
    'bakery': 0.25, 'deli': 0.25, 'dry goods pasta': 0.20,
    'canned goods': 0.20, 'breakfast': 0.20, 'pets': 0.20,
    'babies': 0.15, 'dairy eggs': 0.15, 'international': 0.25,
    'bulk': 0.30, 'missing': 0.20, 'other': 0.20
}

# Maintenant 'department' existe, le KeyError va disparaître !
df_final['estimated_margin_rate'] = df_final['department'].map(margins_config).fillna(0.20)

# Calcul de la marge nette
df_final['unit_margin_after_promo'] = (df_final['unit_price'] * df_final['estimated_margin_rate']) - \
                                      (df_final['unit_price'] * df_final['discount_rate'])

# Résultat final pour le CSV
df_final['total_net_margin_euro'] = df_final['unit_margin_after_promo'] * \
                                   (df_final['confidence'] + (df_final['incremental_sales'] * 0.7))

# 2. Intégration dans le DataFrame
# Note : Assure-toi que la colonne 'department' est bien présente dans ton df_final
df_final['estimated_margin_rate'] = df_final['department'].map(margins_config).fillna(0.20)

# 3. Calcul de la rentabilité réelle par unité
# Marge brute (avant promo) - Montant du discount donné
df_final['unit_margin_after_promo'] = (df_final['unit_price'] * df_final['estimated_margin_rate']) - \
                                      (df_final['unit_price'] * df_final['discount_rate'])

# 4. Profit Net Réel (Marge Totale tenant compte du volume et de la cannibalisation)
# Formule : (Marge Unitaire Promo) * (Ventes de base + Ventes incrémentales corrigées)
df_final['total_net_margin_euro'] = df_final['unit_margin_after_promo'] * \
                                   (df_final['confidence'] + (df_final['incremental_sales'] * 0.7))

# 5. Logique de décision pour le Directeur
def final_director_verdict(row):
    if row['unit_margin_after_promo'] < 0:
        return "❌ INTERDIT (Vente à perte)"
    elif row['total_net_margin_euro'] < 0.05:
        return "⚠️ RISQUÉ (Marge trop faible)"
    else:
        return "✅ VALIDÉ (Rentable)"

df_final['verdict_directeur'] = df_final.apply(final_director_verdict, axis=1)

csv_export = df_final[[
    'cluster', 'department', 'product_name_right', 'unit_price',
    'discount_value', 'estimated_margin_rate', 'unit_margin_after_promo',
    'total_net_margin_euro', 'verdict_directeur'
]]
csv_export.to_csv("../data/outputs/RAPPORT_DIRECTEUR_RENTABILITE.csv", index=False)

print("💰 Analyse terminée.")
print(f"Nombre de ventes à perte détectées : {len(df_final[df_final['unit_margin_after_promo'] < 0])}")

In [ ]:
# Résumé des décisions pour le Directeur
print(df_final['verdict_directeur'].value_counts())

# Profit total réel généré par la campagne (après toutes corrections)
profit_total = df_final['total_net_margin_euro'].sum()
print(f"💵 Profit Net Additionnel estimé pour le magasin : {profit_total:.2f} €")